# Importing Modules

In [1]:
import cv2
import mediapipe as mp
import threading
import numpy as np
import autopy
from google.protobuf.json_format import MessageToDict
import math

# 2. Draw Hands
<img src=https://i.imgur.com/qpRACer.png />

# Initializing some necessary modules

In [2]:
mp_hands = mp.solutions.hands
lms = mp_hands.HandLandmark #landmarks constants i.e lms.WRIST

In [3]:
hands = mp_hands.Hands()
mp_drawing = mp.solutions.drawing_utils

In [4]:
drawSpecs = mp_drawing.DrawingSpec(thickness=1, circle_radius=2,)

# Model class for xy pixel pair

In [5]:
class PixelXY:
     def __init__(self, x, y):
        self.x = x
        self.y = y
        

# Model class for hand data

In [6]:
class Hand:
    def __init__(self, landmarks_array, label):
        self.landmarks = landmarks_array
        self.label = label

# General model class for and array 

In [7]:
class Object:
    def __self__(self, data):
        self.data = data
        

# Function for converting proportional values to pixel

In [8]:

def get_pixel_xy(x,y,iw,ih):
    a = int(x*iw)
    b = int(y*ih)
    
    return(a,b)


# Function for getting all pixel's xy from list of proportional landmarks values

In [9]:
#returns an array of objects of PixelXY which contains x y pixel locations for all points
def get_all_xy(handLms, iw, ih):
    xy_pixels = []
    for index, lm in enumerate(handLms):
        x,y = get_pixel_xy(lm.x,lm.y,iw,ih)
        xy_pixels.insert(index, PixelXY(x,y)) 
    return xy_pixels

# Function for drawing distance line and text between two points 

In [10]:
def draw_distance(p1, p2, lenth, img, direction=''):
    x1 = p1.x
    x2 = p2.x
    y1 = p1.y
    y2 = p2.y
    
    #central points
    cx , cy = (x1+x2) // 2,  (y1+y2) // 2,
   
    
    if direction == 'x':
        cv2.line(img, (x1,cy), (x2,cy), (0,0,255), 1)
        cv2.circle(img, (x1,cy), 5, (0,255,0), cv2.FILLED) #first point green
        cv2.circle(img, (x2,cy), 5, (0,255,255), cv2.FILLED) #second point yellow
        cv2.circle(img, (cx,cy), 5, (255,255,0), cv2.FILLED) #centre point
    
    if direction == 'y':
        cv2.line(img, (cx,y1), (cx,y2), (0,0,255), 1)
        cv2.circle(img, (cx,y1), 5, (0,255,0), cv2.FILLED) #first point green
        cv2.circle(img, (cx,y2), 5, (0,255,255), cv2.FILLED) #second point yellow
        cv2.circle(img, (cx,cy), 5, (255,255,0), cv2.FILLED) #centre point
        
    if direction =='':
        cv2.line(img, (x1,y1), (x2,y2), (0,0,255), 1)
        cv2.circle(img, (x1,y1), 5, (0,255,0), cv2.FILLED) #first point green
        cv2.circle(img, (x2,y2), 5, (0,255,255), cv2.FILLED) #second point yellow
        cv2.circle(img, (cx,cy), 5, (255,255,0), cv2.FILLED) #centre point
    
    cv2.putText(img, lenth, (cx,cy), cv2.FONT_HERSHEY_COMPLEX_SMALL, 1, (0,0, 255), 1, cv2.LINE_AA)

    return img

# This function tells whether distance is positive or negetive
for right hand distance is positive in right direction
for left hand distance is positive in left direction

In [11]:
def get_distance_sign(x1,x2,y1,y2, hand, orientation):
    dist_sign = ''
    
    #for right hand distance is positive in right direction
    #for left hand distance is positive in left direction
    if hand == "Right":
        dist_sign = '+'
    
        if orientation == 'h':
            if x2 < x1:
                dist_sign = '-'
            else:
                if x2 > x1:
                    dist_sign = '+'
        else:
            if orientation == 'v':
                if y2 > y1:
                    dist_sign = '-'
                else:
                    if y2 < y1:
                        dist_sign = '+'
    
    if hand == "Left":
        dist_sign = '-'

        if orientation == 'h':
            if x2 > x1:
                dist_sign = '-'
            else:
                if x2 < x1:
                    dist_sign = '+'
        else:
            if orientation == 'v':
                if y2 > y1:
                    dist_sign = '-'
                else:
                    if y2 < y1:
                        dist_sign = '+'
    return dist_sign

# This function gives the distance between two points and draws it if 
#draw= True

In [12]:
def distance(p1,p2, img=None, orientation = '', hand='', draw = False):
    x1 = p1.x
    x2 = p2.x
    y1 = p1.y
    y2 = p2.y
    
     #it cares about left and right hand 
     #for right hand distance is positive in right direction
    #for left hand distance is positive in left direction
    dist_sign = ''
    
    if hand:
        dist_sign = get_distance_sign(x1=x1, x2=x2, y1=y1, y2=y2, hand=hand, orientation=orientation)
    
    lenth = math.hypot(x2-x1, y2-y1)
    text_len = dist_sign+str(int(lenth))
    if draw:
        img = draw_distance(p1, p2, text_len,img )
    return lenth, dist_sign, img
    

# Only x distance

In [13]:
def distanceX(p1,p2, img=None, orientation = 'h', hand='', draw = False):
    x1 = p1.x
    x2 = p2.x
    y1 = p1.y
    y2 = p2.y
    
     #it cares about left and right hand 
     #for right hand distance is positive in right direction
    #for left hand distance is positive in left direction
    dist_sign = ''
    
    if hand:
        dist_sign = get_distance_sign(x1=x1, x2=x2, y1=y1, y2=y2, hand=hand, orientation=orientation)
    
    lenth = int(x2-x1)
    text_len = dist_sign+str(int(lenth))
    if draw:
        img = draw_distance(p1, p2, text_len,img, 'x' )
    return np.absolute(lenth), dist_sign, img

# Only Y distance

In [14]:
def distanceY(p1,p2, img=None, orientation = 'v', hand='', draw = False):
    x1 = p1.x
    x2 = p2.x
    y1 = p1.y
    y2 = p2.y
    
     #it cares about left and right hand 
     #for right hand distance is positive in right direction
    #for left hand distance is positive in left direction
    dist_sign = ''
    
#     if hand:
#         dist_sign = get_distance_sign(x1=x1, x2=x2, y1=y1, y2=y2, hand=hand, orientation=orientation)
    
    lenth = int(y1-y2)
    text_len = dist_sign+str(int(lenth))
    if draw:
        img = draw_distance(p1, p2, text_len,img, 'y' )
    return lenth, dist_sign, img

# Detect singal hand sign

In [15]:
def detect_single_hand_sign(hands_data):
    
    gesture = 'none'
    
    label = hands_data[0].label
    points = hands_data[0].landmarks
    
    index_mcp = points[lms.INDEX_FINGER_MCP]
    index_dip = points[lms.INDEX_FINGER_DIP]
    index_tip = points[lms.INDEX_FINGER_TIP]
    
    middle_mcp = points[lms.MIDDLE_FINGER_MCP]
    middle_tip = points[lms.MIDDLE_FINGER_TIP]
    
    ring_mcp = points[lms.RING_FINGER_MCP]
    ring_tip = points[lms.RING_FINGER_TIP]
    
    pinky_mcp = points[lms.PINKY_MCP]
    pinky_tip = points[lms.PINKY_TIP]
    
    dist_mm_mt, sign_mm_mt, img = distanceX(p1=middle_mcp, p2=middle_tip, hand=label )
    dist_rm_rt, sign_rm_rt, img = distanceX(p1=ring_mcp, p2=ring_tip, hand=label )
    dist_pm_pt, sign_pm_pt, img = distanceX(p1=pinky_mcp, p2= pinky_tip, hand=label )
    
    dist_im_it, sign_im_it, img = distanceX(p1=index_mcp, p2=index_tip, hand=label )
    dist_id_it, sign_id_it, img = distanceX(p1=index_dip, p2=index_tip, hand=label )
    
    if sign_id_it == '+':
        if sign_mm_mt == '-':
            if sign_rm_rt == '-':
                if sign_pm_pt == '-':
                    if label == 'Left':
                        gesture = 'point left'
                    else:
                        if label == 'Right':
                            gesture = 'point right'
    
    
    
    return gesture
                

# Detects if Zoom mode

In [16]:
def detect_zoom(hands_data, img=None, draw=True ):
    index_tip1 = total_hands_data[0].landmarks[lms.INDEX_FINGER_TIP]
    index_tip2 = total_hands_data[1].landmarks[lms.INDEX_FINGER_TIP]
                
    thumb_tip1 = total_hands_data[0].landmarks[lms.THUMB_TIP]
    thumb_tip2 = total_hands_data[1].landmarks[lms.THUMB_TIP]
                
    
    zoom_dist ,b, img = distance(p1=index_tip1, p2=index_tip2, orientation='h', img=img, draw=draw)
    dist1, sign1, img = distance(p1=thumb_tip1, p2=index_tip1, img=img, draw=draw)
    dist2, sign2, img = distance(p1=thumb_tip2, p2=index_tip2, img=img, draw=draw)
    
    
    threshold = 30
    zoom_indicator = False
    
    if dist1 < threshold and dist2 < threshold:
        zoom_indicator = True
    else:
        zoom_indicator = False
    
    return zoom_indicator, zoom_dist, img
    
    

# Draw Connection

In [17]:
def draw_connections(img, hand_lms):
    mp_drawing.draw_landmarks(img, hand_lms, mp_hands.HAND_CONNECTIONS, 
                                        mp_drawing.DrawingSpec(color=(121, 22, 76), thickness=3, circle_radius=5),
                                        mp_drawing.DrawingSpec(color=(250, 44, 250), thickness=2, circle_radius=2),
                                         )
    return img

# Main Program, Main loop that loops throug each frame

In [18]:
cap = cv2.VideoCapture(0)
sc_w, sc_h = autopy.screen.size()

prev_X, prev_Y = 0,0
cur_X, cur_Y = 0,0
flag = True

while True:
    
    dumpImg = np.zeros((720,1280,3))
    success, img = cap.read()
    
    if success:
        img = cv2.flip(img, 1)
        imgRGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ih, iw, ic = img.shape
        results = hands.process(imgRGB)
        
        if results.multi_hand_landmarks:
            total_hands_data = [] #array of Hand objects contains data about all the hands in the frame

             #custom code       
            zipped = zip(results.multi_hand_landmarks,results.multi_handedness)
            for hand_lms, classification in list(zipped):
                
                #rendering connections
                img = draw_connections(img, hand_lms)
                
                points = get_all_xy(hand_lms.landmark, iw, ih)
                
                # Wrist XY
                x = points[lms.WRIST].x
                y = points[lms.WRIST].y
                hand_label = classification.classification[0].label
                
                total_hands_data.append(Hand(points, hand_label))#storing hands data for universal use
                
                #displaying label for hand at wrist location
                cv2.putText(img, hand_label, (x,y+20), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2, cv2.LINE_AA)
                
                
#                 dist,sign, img = distance(p1=points[lms.THUMB_TIP], p2=points[lms.INDEX_FINGER_TIP], orientation='v', img=img, draw=True, hand = hand_label)
                #drawing verticle line on hands
                hand_hight, s, img = distance(p1=points[lms.WRIST], p2=points[lms.MIDDLE_FINGER_TIP], orientation='v', img=img, draw=True, hand = hand_label)
                
                
                #for pixel in xy_pixels:
                    #cv2.putText(img, str(pixel.x), (pixel.x, pixel.y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 1, cv2.LINE_AA)
        
            
#-----------endfor hand_lms, classification in list(zipped)
            total_hands = len(total_hands_data)
            cv2.putText(img, str(total_hands), (30, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)
            if total_hands == 1:
                #single hand
                gesture = detect_single_hand_sign(total_hands_data)
                cv2.putText(img, gesture, (30, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2, cv2.LINE_AA)
                
            if total_hands == 2:
                #drawing middle tip to tip distance for debugging
                a,b,img = distance(p1=total_hands_data[0].landmarks[lms.MIDDLE_FINGER_TIP], p2=total_hands_data[1].landmarks[lms.MIDDLE_FINGER_TIP], orientation='h', img=img, draw=True, hand = hand_label)
                zoom_indicator, zoom_level, img = detect_zoom(total_hands_data, img, False)
                if zoom_indicator:
                    cv2.putText(img, "zoom mode: "+str(int(zoom_level)), (60, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2, cv2.LINE_AA)
                    
                               
                
            
            
            
#--------endif multilandmarks   
        
        cv2.imshow("image", img)
                
#----endif success
    else:
        print("failed")

 
    cv2.waitKey(1)


failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed
failed

KeyboardInterrupt: 

In [ ]:
total_hands_data[0].label


In [48]:
points = get_all_xy(hand_lms.landmark, iw, ih)
for index, point in enumerate(points):
    text = str(index)+str('= x: ')+str(point.x)+str(' || y: ')+str(point.y)
    print(text)
   

0= x: 360 || y: 272
1= x: 365 || y: 237
2= x: 378 || y: 211
3= x: 393 || y: 194
4= x: 406 || y: 182
5= x: 413 || y: 218
6= x: 436 || y: 211
7= x: 453 || y: 207
8= x: 471 || y: 204
9= x: 421 || y: 235
10= x: 444 || y: 227
11= x: 429 || y: 228
12= x: 415 || y: 230
13= x: 426 || y: 251
14= x: 448 || y: 247
15= x: 436 || y: 246
16= x: 425 || y: 247
17= x: 427 || y: 269
18= x: 448 || y: 268
19= x: 443 || y: 265
20= x: 433 || y: 265


In [198]:
zipped = zip(results.multi_hand_landmarks,results.multi_handedness)
for x,y in list(zipped):
    print(x.landmark[20].x, y.classification[0].label)
    

0.5337216854095459 Left
0.9701544642448425 Right


In [37]:
d = {'a' : 3, 'b': 4}

In [38]:
d['a']

3

In [ ]:
 x_ = np.interp(x, (20,iw-20), (0,sc_w))
                y_ = np.interp(y, (20,ih-40), (0,sc_h))
                
                cur_X = x_
                cur_Y = y_
                
                if cur_X - prev_X > 2 or cur_Y - prev_Y > 2:
                    if sc_w-x_<sc_w and y_<sc_h:
                        autopy.mouse.move((sc_w - x_), y_)
                        text = str(int(x_))+" : "+str(int(y_))
                        cv2.putText(img,text ,(20,400), cv2.FONT_HERSHEY_COMPLEX,1, (0,0,255),2)    
                prev_X = cur_X
                prev_Y = cur_Y

In [5]:
cap = cv2.VideoCapture(0)
def hand_thread():
    while True:
        success, img = cap.read()
        if success:
            imgRGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            result = hands.process(imgRGB)

            if result.multi_hand_landmarks:
                for handLMS in result.multi_hand_landmarks:
                    draw.draw_landmarks(img,handLMS, mHand.HAND_CONNECTIONS)
            cv2.imshow("image", img)
        else:
            print("failed")


        cv2.waitKey(1)
        
        
def face_thread():
    while True:
        success, img = cap.read()
        if success:
            imgRGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            result = faces.process(imgRGB)

            if result.multi_face_landmarks:
                for faceLMS in result.multi_face_landmarks:
                    draw.draw_landmarks(img,faceLMS, mFace.FACE_CONNECTIONS, drawSpecs, drawSpecs)


            cv2.imshow("image", img)
        else:
            print("failed")


        cv2.waitKey(1)

In [5]:
cap = cv2.VideoCapture(0)
while True:
        success, img = cap.read()
        if success:
            imgRGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            result = hands.process(imgRGB)
            
            if result.multi_hand_landmarks:
                for handLMS in result.multi_hand_landmarks:
                    draw.draw_landmarks(img,handLMS, mHand.HAND_CONNECTIONS)
                    
            if result_2.multi_face_landmarks:
                for faceLMS in result_2.multi_face_landmarks:
                    draw.draw_landmarks(img,faceLMS, mFace.FACE_CONNECTIONS, drawSpecs, drawSpecs)
                    
            cv2.imshow("image", img)
        else:
            print("failed")


        cv2.waitKey(1)
        


KeyboardInterrupt: 